# 추가 백엔드-4 (Vision 1/2회/3회 이상 FAIL 분석)

본 노트북은 **`2026-01-29 추가 백엔드-4 설계`** 사양에 맞춰,
- KST 기준 주/야간 생산 윈도우를 만들고
- `a3_vision_table.vision_table` 에서 해당 시간대의 `result='FAIL'` 만 스캔한 뒤
- 바코드(18번째 문자)로 품번(pn) 및 PD/Non-PD를 분류하고
- 바코드별 FAIL 발생 횟수(1회 / 2회 / 3회 이상)를 기준으로 **1st/2st/3st_over** DataFrame 3개를 출력합니다.

> **해석(중요)**: `2회 발생`/`3회 이상 발생`의 step_description은 **바코드별 FAIL을 end_ts 기준으로 정렬**한 뒤,
> 각각 **2번째 FAIL row / 3번째 FAIL row** 의 `step_description`을 대표값으로 사용합니다.


In [1]:
# 필요 패키지 설치(필요 시)
# !pip -q install sqlalchemy psycopg2-binary pandas python-dateutil

In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime, date, timedelta
from typing import Tuple, Dict

# =========================
# 1) DB 설정 (사양 그대로)
# =========================
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

def get_engine():
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    # 운영 요청(풀 최소화): pool_size=1, max_overflow=0
    return create_engine(url, pool_size=1, max_overflow=0, pool_pre_ping=True)

engine = get_engine()
print("[OK] engine ready")

[OK] engine ready


In [3]:
# =========================
# 2) 주/야간 윈도우 (KST 로컬 기준)
# =========================
# day   : [D-DAY] 08:30:00 ~ 20:29:59 (inclusive)
# night : [D-DAY] 20:30:00 ~ 23:59:59 + [D+1] 00:00:00 ~ 08:29:59 (inclusive)

def shift_window(prod_day: str, shift_type: str) -> Tuple[datetime, datetime]:
    """prod_day: YYYYMMDD, shift_type: 'day'|'night'. 반환은 naive datetime(KST 로컬 해석)."""
    if len(prod_day) != 8 or not prod_day.isdigit():
        raise ValueError("prod_day must be YYYYMMDD")
    d0 = date(int(prod_day[:4]), int(prod_day[4:6]), int(prod_day[6:8]))
    if shift_type == "day":
        start = datetime(d0.year, d0.month, d0.day, 8, 30, 0)
        end   = datetime(d0.year, d0.month, d0.day, 20, 29, 59)
    elif shift_type == "night":
        start = datetime(d0.year, d0.month, d0.day, 20, 30, 0)
        d1 = d0 + timedelta(days=1)
        end   = datetime(d1.year, d1.month, d1.day, 8, 29, 59)
    else:
        raise ValueError("shift_type must be 'day' or 'night'")
    return start, end

# ===== 사용자 입력 =====
PROD_DAY = "20260127"      # YYYYMMDD
SHIFT    = "day"           # 'day' or 'night'

start_dt, end_dt = shift_window(PROD_DAY, SHIFT)
print("[WINDOW]", PROD_DAY, SHIFT, start_dt, "->", end_dt)

[WINDOW] 20260127 day 2026-01-27 08:30:00 -> 2026-01-27 20:29:59


In [4]:
# =========================
# 3) remark_info 매핑 로드
# =========================
# 사양: g_production_film.remark_info
# barcode_information(=18번째 문자 key), pn, remark

def load_remark_map(engine) -> Dict[str, Dict[str, str]]:
    """key_char -> {pn:str, remark:str} 형태로 반환"""
    sql = text("""
        SELECT barcode_information, pn, remark
        FROM g_production_film.remark_info
    """)
    try:
        df = pd.read_sql(sql, engine)
        mp = {}
        for _, r in df.iterrows():
            k = str(r["barcode_information"]).strip()
            mp[k] = {"pn": str(r["pn"]).strip(), "remark": str(r["remark"]).strip()}
        if not mp:
            raise RuntimeError("remark_info empty")
        return mp
    except Exception as e:
        # DB 접근 실패/스키마 불일치 시, PDF 표(사양서) 기반 fallback
        print("[WARN] remark_info load failed -> fallback dict. reason:", e)
        return {
            "C": {"pn": "35643009", "remark": "Non-PD"},
            "P": {"pn": "35643010", "remark": "Non-PD"},
            "1": {"pn": "35654264", "remark": "Non-PD"},
            "N": {"pn": "35749091", "remark": "Non-PD"},
            "J": {"pn": "35930927", "remark": "PD"},
            "S": {"pn": "35930928", "remark": "PD"},
        }

remark_map = load_remark_map(engine)
remark_map

{'C': {'pn': '35643009', 'remark': 'Non-PD'},
 'P': {'pn': '35643010', 'remark': 'Non-PD'},
 '1': {'pn': '35654264', 'remark': 'Non-PD'},
 'N': {'pn': '35749091', 'remark': 'Non-PD'},
 'J': {'pn': '35930927', 'remark': 'PD'},
 'S': {'pn': '35930928', 'remark': 'PD'}}

In [5]:
# =========================
# 4) Vision FAIL 원천 데이터 로드
# =========================
# 스키마: a3_vision_table
# 테이블: vision_table
# 컬럼: end_day(text:YYYYMMDD), end_time(text:HH:MI:SS[.fff]), station, barcode_information, step_description, result

def fetch_vision_fails(engine, start_dt: datetime, end_dt: datetime) -> pd.DataFrame:
    # end_time 정규화: '.' 앞 8글자 사용 (HH:MI:SS)
    sql = text("""
        WITH base AS (
            SELECT
                station,
                barcode_information,
                step_description,
                end_day,
                end_time,
                -- normalize HH:MI:SS
                split_part(end_time, '.', 1) AS end_time_norm
            FROM a3_vision_table.vision_table
            WHERE result = 'FAIL'
        ), ts AS (
            SELECT
                station,
                barcode_information,
                step_description,
                end_day,
                end_time_norm,
                to_timestamp(end_day || ' ' || end_time_norm, 'YYYYMMDD HH24:MI:SS') AS end_ts
            FROM base
        )
        SELECT *
        FROM ts
        WHERE end_ts >= :start_dt AND end_ts <= :end_dt
    """)
    df = pd.read_sql(sql, engine, params={"start_dt": start_dt, "end_dt": end_dt})
    return df

df_raw = fetch_vision_fails(engine, start_dt, end_dt)
print("[ROWS]", len(df_raw))
df_raw.head()

[ROWS] 16


,station,barcode_information,step_description,end_day,end_time_norm,end_ts
0,Vision1,BA1WJ26027501331PC3T-14F014-AC,intensity6,20260127,08:33:44,2026-01-26 23:33:44+00:00
1,Vision1,BA1WJ26027501337PC3T-14F014-AC,intensity6,20260127,08:36:02,2026-01-26 23:36:02+00:00
2,Vision1,BA1WJ26027502495PC3T-14F014-AC,intensity6,20260127,09:05:05,2026-01-27 00:05:05+00:00
3,Vision2,BA1WJ26027504073PC3T-14F014-AC,intensity2,20260127,12:53:59,2026-01-27 03:53:59+00:00
4,Vision2,BA1WJ26027504073PC3T-14F014-AC,intensity5,20260127,12:53:59,2026-01-27 03:53:59+00:00


In [6]:
# =========================
# 5) 바코드 18번째 문자로 PN/Remark 분류
# =========================
def barcode_key_char(barcode: str) -> str:
    if barcode is None:
        return ""
    s = str(barcode)
    return s[17] if len(s) >= 18 else ""

df = df_raw.copy()
df["key_char"] = df["barcode_information"].map(barcode_key_char)
df["pn"] = df["key_char"].map(lambda k: remark_map.get(k, {}).get("pn", "UNKNOWN"))
df["remark"] = df["key_char"].map(lambda k: remark_map.get(k, {}).get("remark", "UNKNOWN"))

# prod_day/shift_type 컬럼 추가
df["prod_day"] = PROD_DAY
df["shift_type"] = SHIFT

df[["station","barcode_information","key_char","pn","remark","step_description","end_ts"]].head()

,station,barcode_information,key_char,pn,remark,step_description,end_ts
0,Vision1,BA1WJ26027501331PC3T-14F014-AC,C,35643009,Non-PD,intensity6,2026-01-26 23:33:44+00:00
1,Vision1,BA1WJ26027501337PC3T-14F014-AC,C,35643009,Non-PD,intensity6,2026-01-26 23:36:02+00:00
2,Vision1,BA1WJ26027502495PC3T-14F014-AC,C,35643009,Non-PD,intensity6,2026-01-27 00:05:05+00:00
3,Vision2,BA1WJ26027504073PC3T-14F014-AC,C,35643009,Non-PD,intensity2,2026-01-27 03:53:59+00:00
4,Vision2,BA1WJ26027504073PC3T-14F014-AC,C,35643009,Non-PD,intensity5,2026-01-27 03:53:59+00:00


In [7]:
# =========================
# 6) 바코드별 FAIL 횟수(1/2/3+) 분류 + 대표 step_description 선정
# =========================
# 규칙(노트북 상단 설명 참조):
# - n=1  -> 1번째 FAIL row의 step_description
# - n=2  -> 2번째 FAIL row의 step_description
# - n>=3 -> 3번째 FAIL row의 step_description

df_sorted = df.sort_values(["barcode_information", "station", "end_ts"], ascending=True)

grp_cols = ["barcode_information", "station", "pn"]
df_sorted["fail_idx"] = df_sorted.groupby(grp_cols).cumcount() + 1
df_sorted["fail_n"] = df_sorted.groupby(grp_cols)["fail_idx"].transform("max")

def pick_row(sub: pd.DataFrame) -> pd.DataFrame:
    n = int(sub["fail_n"].iloc[0])
    if n == 1:
        picked = sub[sub["fail_idx"] == 1].head(1).copy()
        picked["fail_class"] = "1st"
    elif n == 2:
        picked = sub[sub["fail_idx"] == 2].head(1).copy()
        picked["fail_class"] = "2st"
    else:
        picked = sub[sub["fail_idx"] == 3].head(1).copy()
        picked["fail_class"] = "3st_over"
    return picked

picked = (
    df_sorted.groupby(grp_cols, as_index=False, group_keys=False)
    .apply(pick_row)
)

picked[["barcode_information","station","pn","fail_n","fail_class","step_description","end_ts"]].head(10)

C:\Users\user\AppData\Local\Temp\ipykernel_11560\2591072755.py:30: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(pick_row)


,barcode_information,station,pn,fail_n,fail_class,step_description,end_ts
0,BA1WJ26027501331PC3T-14F014-AC,Vision1,35643009,1,1st,intensity6,2026-01-26 23:33:44+00:00
1,BA1WJ26027501337PC3T-14F014-AC,Vision1,35643009,1,1st,intensity6,2026-01-26 23:36:02+00:00
2,BA1WJ26027502495PC3T-14F014-AC,Vision1,35643009,1,1st,intensity6,2026-01-27 00:05:05+00:00
8,BA1WJ26027503774PC3T-14F014-AC,Vision1,35643009,1,1st,intensity6,2026-01-27 02:54:36+00:00
6,BA1WJ26027504073PC3T-14F014-AC,Vision2,35643009,4,3st_over,intensity2,2026-01-27 10:18:15+00:00
5,BA1WJ26027504459PC3T-14F014-AC,Vision2,35643009,2,2st,intensity7,2026-01-27 10:17:07+00:00
13,BA1WJ26027504531PC3T-14F014-AC,Vision1,35643009,1,1st,intensity6,2026-01-27 09:09:38+00:00
11,BA1WJ26027504779PC3T-14F014-AC,Vision1,35643009,1,1st,intensity6,2026-01-27 05:26:23+00:00
10,BA1WJ26027504781PC3T-14F014-AC,Vision1,35643009,1,1st,intensity6,2026-01-27 05:25:53+00:00
14,BA1WJ26027505266PC3T-14F014-AC,Vision2,35643009,1,1st,intensity7,2026-01-27 09:32:49+00:00


In [8]:
# =========================
# 7) 집계 DataFrame 3개 생성
# =========================
now_ts = datetime.now()  # 로컬(=KST 가정) 타임스탬프

agg_cols = ["prod_day", "shift_type", "pn", "station", "step_description"]

agg = (
    picked.groupby(["fail_class"] + agg_cols)["barcode_information"]
    .nunique()
    .reset_index(name="count")
)
agg["updated_at"] = now_ts

df_1st = (
    agg[agg["fail_class"] == "1st"]
    .drop(columns=["fail_class"])
    .rename(columns={"step_description": "1st_FAIL_step_description"})
    .sort_values(["pn","station","count"], ascending=[True, True, False])
)

df_2st = (
    agg[agg["fail_class"] == "2st"]
    .drop(columns=["fail_class"])
    .rename(columns={"step_description": "2st_FAIL_step_description"})
    .sort_values(["pn","station","count"], ascending=[True, True, False])
)

df_3st = (
    agg[agg["fail_class"] == "3st_over"]
    .drop(columns=["fail_class"])
    .rename(columns={"step_description": "3st_over_FAIL_step_description"})
    .sort_values(["pn","station","count"], ascending=[True, True, False])
)

print("[1st] rows:", len(df_1st))
print("[2st] rows:", len(df_2st))
print("[3st_over] rows:", len(df_3st))

df_1st.head()

[1st] rows: 4
[2st] rows: 1
[3st_over] rows: 1


,prod_day,shift_type,pn,station,1st_FAIL_step_description,count,updated_at
1,20260127,day,35643009,Vision1,intensity6,7,2026-01-29 18:16:25.668422
0,20260127,day,35643009,Vision1,intensity2,1,2026-01-29 18:16:25.668422
2,20260127,day,35643009,Vision1,intensity8,1,2026-01-29 18:16:25.668422
3,20260127,day,35643009,Vision2,intensity7,1,2026-01-29 18:16:25.668422


In [9]:
# 1st FAIL 결과
df_1st

,prod_day,shift_type,pn,station,1st_FAIL_step_description,count,updated_at
1,20260127,day,35643009,Vision1,intensity6,7,2026-01-29 18:16:25.668422
0,20260127,day,35643009,Vision1,intensity2,1,2026-01-29 18:16:25.668422
2,20260127,day,35643009,Vision1,intensity8,1,2026-01-29 18:16:25.668422
3,20260127,day,35643009,Vision2,intensity7,1,2026-01-29 18:16:25.668422


In [10]:
# 2st FAIL 결과
df_2st

,prod_day,shift_type,pn,station,2st_FAIL_step_description,count,updated_at
4,20260127,day,35643009,Vision2,intensity7,1,2026-01-29 18:16:25.668422


In [11]:
# 3st_over FAIL 결과
df_3st

,prod_day,shift_type,pn,station,3st_over_FAIL_step_description,count,updated_at
5,20260127,day,35643009,Vision2,intensity2,1,2026-01-29 18:16:25.668422


## 참고/검증 팁

- `df_raw`에서 `end_ts`가 **윈도우 범위 내**로 잘 잘리는지 확인하세요.
- 특정 바코드 하나를 골라 `df_sorted[df_sorted['barcode_information']==...].sort_values('end_ts')` 로
  FAIL 순서가 기대대로인지 검증하면 `2st/3st_over` 대표 step_description 해석이 맞는지 빠르게 확인할 수 있습니다.
